# Observability & Debugging — Part 1: Tracing Setup

This notebook teaches you how to configure OpenTelemetry tracing with Strands Agents
using the built-in `StrandsTelemetry` class and run your first traced agent invocation.

**What you'll learn:**
- Configure `StrandsTelemetry` with the console exporter
- Run an agent and observe trace output in stdout
- Capture spans programmatically for inspection

**Prerequisites:**
- Python 3.10+
- AWS credentials configured (for Bedrock model access)
- `pip install -r requirements.txt` completed

## How It Works

`StrandsTelemetry` manages the global OpenTelemetry `TracerProvider`. Once configured,
every `Agent` instance automatically picks up the tracer — no explicit wiring needed.

```
StrandsTelemetry()              →  creates TracerProvider
  .setup_console_exporter()     →  attaches ConsoleSpanExporter
  .setup_otlp_exporter()        →  attaches OTLPSpanExporter

Agent(...)                      →  reads global TracerProvider
  agent("prompt")               →  emits spans automatically
```

**Important:** `StrandsTelemetry` must be configured *before* creating any `Agent`.

In [ ]:
from strands.telemetry.config import StrandsTelemetry

# Configure telemetry with console exporter
# This prints every span to stdout as it completes
telemetry = StrandsTelemetry()
telemetry.setup_console_exporter()

print("✓ Telemetry configured with console exporter")
print("  Every agent invocation will now produce trace output below.")

## Your First Traced Invocation

Let's define a simple tool and create an agent. When we invoke the agent, the console
exporter will print each span as it completes.

In [ ]:
from strands import Agent, tool
from strands.models.bedrock import BedrockModel


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression.

    Args:
        expression: A mathematical expression to evaluate (e.g., "42 * 17")

    Returns:
        The result of the expression as a string.
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Create agent — automatically picks up the global tracer
agent = Agent(
    model=BedrockModel(model_id="us.amazon.nova-lite-v1:0"),
    tools=[calculator],
)

print("✓ Agent created with calculator tool")

In [ ]:
# Invoke the agent — trace spans will print to stdout
result = agent("What is 42 multiplied by 17?")
print(f"\n{'='*60}")
print(f"Agent response: {result}")
print(f"{'='*60}")
print("\n↑ The trace output above shows every span that was created.")
print("  Look for: Agent span, Cycle span(s), Model Invoke span(s), Tool span(s)")

## Capturing Spans Programmatically

The console exporter is great for visual inspection. For programmatic analysis,
we can create a simple span collector that stores spans in a list.

In [ ]:
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult
from opentelemetry import trace


class SpanCollector(SpanExporter):
    """Simple in-memory span collector for tutorial use."""

    def __init__(self):
        self._spans = []

    def export(self, spans):
        self._spans.extend(spans)
        return SpanExportResult.SUCCESS

    def get_finished_spans(self):
        return list(self._spans)

    def clear(self):
        self._spans = []

    def shutdown(self):
        self._spans = []


# Add span collector alongside the console exporter
span_collector = SpanCollector()
provider = trace.get_tracer_provider()
if hasattr(provider, "add_span_processor"):
    provider.add_span_processor(SimpleSpanProcessor(span_collector))
    print("✓ Span collector added — spans will be captured for inspection")
else:
    print("⚠️  Could not add span processor")

In [ ]:
# Clear previous spans and run a fresh invocation
span_collector.clear()
_ = agent("What is 7 + 3?")

# Inspect captured spans
spans = span_collector.get_finished_spans()
print(f"\n📊 Captured {len(spans)} spans:\n")

for span in spans:
    attrs = span.attributes or {}
    duration_ms = (
        (span.end_time - span.start_time) / 1_000_000
        if span.end_time and span.start_time
        else 0
    )
    print(f"  {span.name:<25} duration={duration_ms:.0f}ms  status={span.status.status_code.name}")

## Troubleshooting

| Problem | Solution |
|---------|----------|
| No trace output visible | Console exporter prints spans when they *end*. Wait for the agent call to complete. |
| `ModuleNotFoundError: No module named 'opentelemetry'` | Run `pip install -r requirements.txt` |
| `No TracerProvider configured` | Ensure `StrandsTelemetry` is configured *before* creating the Agent |
| AWS credentials error | Run `aws configure` or set `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` |

## Next

Continue to [02_trace_hierarchy.ipynb](02_trace_hierarchy.ipynb) to understand the
Agent → Cycle → Model → Tool span hierarchy in detail.